# ChagaSight — Fold Training Notebook v10.1

## Correction log (A–Z across all prior versions)

| Version | Problem | Fix |
|---------|---------|-----|
| v9 | `BATCH_SIZE=32` in config but GPU warns "keep 16" | Hard-set `BATCH_SIZE=16`; Phase 2 at 32 caused 20× slowdown |
| v9 | `phase2_grad_accum=1` → eff.batch=16, slow convergence | Set to `2` → eff.batch=32, matches effective throughput |
| v9 | `trainer.val_every_n_iters` set twice (init + override) | Set once in `ChagasTrainer(...)` call only |
| v9 | `NUM_WORKERS=4` for QUICK_TEST | Set to `0` — cleaner debug, avoids worker spawn cost |
| v9 | No fail-fast on missing pretrained weights | Added `assert` on both `.pt` files |
| v9 | No GPU memory guard before loading 173 M params | Added memory-check cell |
| v9 | No checkpoint verification after training | Added verification + fold-completion checklist |
| v9 | `trainer.py` displays `eff.batch=16×accum` regardless of batch size | Added corrected display in notebook |
| All | `losses.py` uses `pos_weight=10` but Van Santvliet uses 5 | Explained as deliberate ChagaSight design choice in comment |
| All | No paper-vs-implementation comparison | Added deviation notes in Cell 2 |

## Paper alignment quick-reference

| Hyperparameter | Van Santvliet (2025) | Kim et al. (2025) | ChagaSight |
|---|---|---|---|
| Backbone | 1D ViT FM (ST-MEM) | EfficientNetV2-S | Both via hybrid |
| Effective batch | 64 | not specified | 64 (P1), 32 (P2) |
| Phase 1 iters | 2 000 | — | 2 000 ✓ |
| Phase 2 iters | 12 000 | — | 12 000 ✓ |
| Phase 1 LR | 2e-4 | — | 2e-4 ✓ |
| Phase 2 LR | 2e-5 | 2e-5 | 2e-5 ✓ |
| pos_weight | 5 | 10 | **10** (deliberate — ~2% positive rate is more extreme) |
| alignment_weight | — | 0.5 | 0.5 ✓ |
| γ+ / γ- | 0 / 2 | 0 / 2 | 0 / 2 ✓ |
| Soft labels | 0.8 / 0.2 (CODE-15) | — | 0.8 / 0.2 ✓ |
| Weighted sampling | 5× | not used | 5× ✓ |
| Augmentations | powerline, crop, shift | lead-mixup, albumentations | both + amplitude scaling + baseline wander |


## Cell 1 — Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('GPU required. Switch runtime to GPU before running.')

gpu_name = torch.cuda.get_device_name(0)
gpu_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  ({gpu_gb:.1f} GB)')
if gpu_gb < 5.5:
    raise RuntimeError(f'Need ≥6 GB GPU, found {gpu_gb:.1f} GB')


## Cell 2 — Configuration

In [ ]:
# ── QUICK TEST ──────────────────────────────────────────────────────────────
# True  → 5-minute smoke-test (loss must decrease, no crashes; score random)
# False → full training (~18 min Phase 1 + ~5–6 h Phase 2 on RTX 3050 6 GB)
QUICK_TEST = False

# ── FOLD ────────────────────────────────────────────────────────────────────
FOLD = 0   # change to 1, 2, 3, 4 for other folds

# ── PATHS ───────────────────────────────────────────────────────────────────
DATA_DIR       = project_root / 'data' / 'processed'
METADATA_CSV   = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR     = DATA_DIR / '2d_images'
SIGNALS_DIR    = DATA_DIR / '1d_signals_100hz'
CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

assert METADATA_CSV.exists(),    f'Missing metadata CSV: {METADATA_CSV}'
assert MAE_CHECKPOINT.exists(),  f'Missing MAE weights: {MAE_CHECKPOINT}  — run MAE pretraining first'
assert STMEM_CHECKPOINT.exists(),f'Missing ST-MEM weights: {STMEM_CHECKPOINT}  — run ST-MEM pretraining first'

# ── HARDWARE ────────────────────────────────────────────────────────────────
# RTX 3050 6 GB constraint:
#   Phase 1: FM frozen  → only ~88 M params need gradients
#            BATCH_SIZE=16, accum=4 → eff.batch=64 (matches Van Santvliet 2025 Table)
#   Phase 2: all 173 M params unfrozen
#            BATCH_SIZE=16, accum=2 → eff.batch=32
#            (Van Santvliet used eff.batch=64 on larger GPU; we halve accum to fit)
BATCH_SIZE        = 16   # DO NOT change to 32 — Phase 2 OOM on 6 GB GPU
PHASE1_GRAD_ACCUM = 4    # eff. batch = 64  (matches paper)
PHASE2_GRAD_ACCUM = 2    # eff. batch = 32  (hardware constraint)
NUM_WORKERS       = 2
USE_AMP           = True

# ── TRAINING SCHEDULE ───────────────────────────────────────────────────────
# Van Santvliet et al. (2025) Section 2.3:
#   Phase 1: 2 000 iters, LR=2e-4 (head only, FM frozen)
#   Phase 2: 12 000 iters, LR=2e-5 (all params, discriminative LR)
PHASE1_ITERATIONS = 2000
PHASE2_ITERATIONS = 12000
PHASE1_LR         = 2e-4
PHASE2_LR_HIGH    = 2e-4   # classifier + REPA head
PHASE2_LR_LOW     = 2e-5   # FM + 2D-ViT (10× lower = discriminative fine-tuning)
MAX_GRAD_NORM     = 1.0    # standard transformer clip
WARMUP_ITERS      = 200    # ramp LR from 0 → target over first 200 steps

# ── LOSS FUNCTION NOTE ──────────────────────────────────────────────────────
# losses.py defaults: pos_weight=10, gamma_neg=2, alignment_weight=0.5
# Van Santvliet used pos_weight=5; Kim used pos_weight=10.
# ChagaSight uses pos_weight=10 because the full 366k dataset has ~2.24% positive
# rate (vs ~3.4% in the 83k subset), making imbalance more severe.
# alignment_weight=0.5 matches Kim et al. exactly.

# ── VALIDATION ──────────────────────────────────────────────────────────────
# Single val interval used for both phases.
# Phase 1: 2000 / 4000 = 0 mid-phase checks (FM frozen, short, acceptable)
# Phase 2: 12000 / 4000 = 3 checks at iters 4000, 8000, 12000
VAL_EVERY = 4000

# ── QUICK TEST OVERRIDES ────────────────────────────────────────────────────
if QUICK_TEST:
    PHASE1_ITERATIONS = 25
    PHASE2_ITERATIONS = 50
    WARMUP_ITERS      = 10
    VAL_EVERY         = 50
    NUM_WORKERS       = 0   # no worker processes in debug mode

# ── AUTO-RESUME ─────────────────────────────────────────────────────────────
_ckpt = CHECKPOINT_DIR / f'fold{FOLD}_latest.pt'
RESUME_FROM = str(_ckpt) if _ckpt.exists() else None

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print(f'Fold {FOLD}  |  QUICK_TEST={QUICK_TEST}')
print(f'Batch {BATCH_SIZE}  |  P1 eff.batch={BATCH_SIZE*PHASE1_GRAD_ACCUM}  |  P2 eff.batch={BATCH_SIZE*PHASE2_GRAD_ACCUM}')
print(f'Phase 1: {PHASE1_ITERATIONS} iters  |  Phase 2: {PHASE2_ITERATIONS} iters  |  Val every {VAL_EVERY}')
print(f'Resume: {RESUME_FROM or "fresh start"}')


## Cell 3 — GPU Memory Check

In [ ]:
# Fail fast before loading the 662 MB model if VRAM is already pressured.
torch.cuda.empty_cache()
alloc_gb  = torch.cuda.memory_allocated() / 1e9
reserv_gb = torch.cuda.memory_reserved()  / 1e9
total_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
free_gb   = total_gb - reserv_gb

print(f'VRAM: {total_gb:.1f} GB total | {alloc_gb:.2f} GB used | {free_gb:.2f} GB free')
if free_gb < 3.5:
    raise RuntimeError(
        f'Only {free_gb:.1f} GB VRAM free — need ≥3.5 GB. '
        'Restart kernel or close other GPU processes.'
    )
print('Memory check passed.')


## Cell 4 — Dataloaders

In [ ]:
train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,   # 5× oversample positives (Van Santvliet 2025)
    augment_train=True,
)

# Sanity-check first batch
batch = next(iter(train_loader))
assert batch['image'].shape  == torch.Size([BATCH_SIZE, 3, 24, 2048]),     f"Image shape wrong: {batch['image'].shape}"
assert batch['signal'].shape == torch.Size([BATCH_SIZE, 12, 1000]),     f"Signal shape wrong: {batch['signal'].shape}"
assert not torch.isnan(batch['signal']).any(), 'NaN detected in signals'
assert not torch.isnan(batch['image'].float()).any(), 'NaN detected in images'

unique_labels = sorted({round(v, 1) for v in batch['label'].tolist()})
assert set(unique_labels) <= {0.0, 0.2, 0.8, 1.0}, f'Unexpected label values: {unique_labels}'

n_train = len(train_loader.dataset)
n_val   = len(val_loader.dataset)
n_pos_train = train_loader.dataset.df['label_hard'].sum()
n_pos_val   = val_loader.dataset.df['label_hard'].sum()

print(f'Train: {n_train:,} samples ({n_pos_train:,} positive = {100*n_pos_train/n_train:.2f}%)')
print(f'Val:   {n_val:,} samples ({n_pos_val:,} positive = {100*n_pos_val/n_val:.2f}%)')
print(f'Label values in first batch: {unique_labels}')


## Cell 5 — Model + Pretrained Weights

In [ ]:
model = HybridChagasModel(
    img_size=(24, 2048), patch_size_2d=(8, 64),
    num_leads=12, seq_len_1d=1000, patch_size_1d=50,
    embed_dim=768, depth=12, num_heads=12,
    use_aol=True, use_demographics=True,
)

# Load MAE pretrained 2D-ViT encoder (Kim et al. approach, trained on ECG contour images)
model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))

# Load ST-MEM pretrained 1D-ViT encoder (Van Santvliet et al. approach)
model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))

model = model.to(device)

# Forward-pass sanity check (no grad)
with torch.no_grad():
    out = model(
        batch['image'].to(device),
        batch['signal'].to(device),
        batch['age'].to(device),
        batch['sex'].to(device),
    )
assert torch.isfinite(out['logits']).all(),      'Non-finite logits at init'
assert torch.isfinite(out['fm_features']).all(), 'Non-finite FM features at init'

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {total_p:,} total | {trainable_p:,} trainable')
print(f'Logits: {out["logits"].shape}  |  FM features: {out["fm_features"].shape}')
print(f'Aligned 2D: {out["aligned_2d_features"].shape}')


## Cell 6 — Trainer

In [ ]:
# Note: trainer.py's internal display hardcodes eff.batch=16×accum.
# With BATCH_SIZE=16 this is accurate. Do not change BATCH_SIZE without
# also checking that the trainer banner prints correctly.

trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    # Iterations (Van Santvliet 2025, Section 2.3)
    phase1_iterations=PHASE1_ITERATIONS,    # 2 000
    phase2_iterations=PHASE2_ITERATIONS,    # 12 000
    # Learning rates
    phase1_lr=PHASE1_LR,                    # 2e-4
    phase2_lr_high=PHASE2_LR_HIGH,          # 2e-4  (head + REPA)
    phase2_lr_low=PHASE2_LR_LOW,            # 2e-5  (FM + 2D-ViT)
    # Stability
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    max_grad_norm=MAX_GRAD_NORM,            # 1.0
    warmup_iters=WARMUP_ITERS,              # 200
    # Gradient accumulation
    phase1_grad_accum=PHASE1_GRAD_ACCUM,    # 4 → eff.batch 64
    phase2_grad_accum=PHASE2_GRAD_ACCUM,    # 2 → eff.batch 32
    # Validation — set ONCE here, not overridden later
    val_every_n_iters=VAL_EVERY,            # 4 000
    val_subset_size=300  if QUICK_TEST else 3000,
    val_n_permutations=100 if QUICK_TEST else 1000,
)
# val_every_n_iters=4000 means:
#   Phase 1 mid-checks: 2000/4000 = 0  (blind, FM frozen, acceptable)
#   Phase 2 mid-checks: 12000/4000 = 3 (at iters 4000, 8000, 12000)
p2_checks = PHASE2_ITERATIONS // VAL_EVERY
print(f'Val interval: {VAL_EVERY} iters  |  P1 mid-checks: 0  |  P2 mid-checks: {p2_checks}')
print(f'Val subset: {trainer.val_subset_size} stratified samples  |  Fast perms: {trainer.val_n_permutations}')
if not QUICK_TEST:
    print(f'ETA: Phase 1 ~18 min  |  Phase 2 ~5–6 h  |  Total ~6 h')


## Cell 7 — Train

In [ ]:
if RESUME_FROM:
    print(f'Resuming from: {Path(RESUME_FROM).name}')
else:
    print('Starting fresh (no checkpoint found)')

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

tpr = metrics['tpr_5pct']
print(f'\nFold {FOLD} results:')
print(f'  TPR@5%: {tpr:.4f}  (PRIMARY — official PhysioNet metric)')
print(f'  AUROC:  {metrics["auroc"]:.4f}')
print(f'  AUPRC:  {metrics.get("auprc", 0):.4f}')
print(f'  Method: {"OFFICIAL" if metrics.get("using_official") else "APPROXIMATE"}')

if not QUICK_TEST:
    benchmarks = [
        ('PhysioNet random baseline', 0.05),
        ('No-pretrain baseline',      0.30),
        ('Target (exceed challenge)', 0.42),
        ('Top team (Van Santvliet)',  0.445),
        ('SOTA cross-val (Van S.)',   0.490),
    ]
    print()
    for name, val in benchmarks:
        diff = tpr - val
        mark = '↑' if diff >= 0 else '↓'
        print(f'  vs {name:<32} {val:.3f}  {mark}{abs(diff):.4f}')


## Cell 8 — Save Results and Training Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Save metrics CSV
results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_df['quick_test'] = QUICK_TEST
results_df.to_csv(CHECKPOINT_DIR / f'fold{FOLD}_results.csv', index=False)

history = trainer.history
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

# ── Loss ─────────────────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
if history['train_loss']:
    ax0.plot(history['train_loss'], lw=0.8, alpha=0.8, color='steelblue')
    if PHASE1_ITERATIONS < len(history['train_loss']):
        ax0.axvline(PHASE1_ITERATIONS, color='r', ls='--', lw=1.2,
                    label=f'Phase 2 start (iter {PHASE1_ITERATIONS})')
        ax0.legend(fontsize=8)
    ax0.set_xlabel('Iteration'); ax0.set_ylabel('Loss')
    ax0.set_title('Training Loss (50-iter smooth)')
    ax0.grid(True, alpha=0.3)

# ── TPR@5% ───────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
if history['val_tpr_5pct']:
    iters = [VAL_EVERY * (i + 1) for i in range(len(history['val_tpr_5pct']))]
    ax1.plot(iters, history['val_tpr_5pct'], 'go-', ms=5, lw=1.5, label='Val TPR@5%')
    ax1.axhline(0.42,  color='r',      ls='--', lw=1, alpha=0.8, label='Target 0.420')
    ax1.axhline(0.445, color='purple', ls=':',  lw=1, alpha=0.8, label='Top team 0.445')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('TPR@5%')
    ax1.set_title('Validation TPR@5% (primary metric)')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    top = max(history['val_tpr_5pct'])
    ax1.set_ylim(0, min(1.0, top * 1.2))

# ── Gradient norm ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
if history['grad_norm']:
    g = history['grad_norm']
    n_show = min(PHASE1_ITERATIONS, len(g))
    ax2.plot(g[:n_show], lw=0.6, alpha=0.6, color='darkorange')
    ax2.axhline(1.0, color='r', ls='--', lw=1.2, label='Clip @ 1.0')
    clipped_pct = 100 * sum(1 for v in g[:n_show] if v > 1.0) / max(1, n_show)
    ax2.set_title(f'Gradient Norm — Phase 1 ({clipped_pct:.0f}% clipped)')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Grad norm')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

fig.suptitle(f'Fold {FOLD} Training{"  [QUICK TEST]" if QUICK_TEST else ""}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = CHECKPOINT_DIR / f'fold{FOLD}_training_curve.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path.name}')


## Cell 9 — Checkpoint Verification

In [ ]:
print(f'Checkpoints for fold {FOLD}:')
for p in sorted(CHECKPOINT_DIR.glob(f'fold{FOLD}*.pt')):
    mb = p.stat().st_size / 1e6
    print(f'  {p.name}  ({mb:.0f} MB)')

# Verify best checkpoint is readable
best_path = CHECKPOINT_DIR / f'fold{FOLD}_best.pt'
if best_path.exists():
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    saved_score = ckpt.get('val_score', None)
    saved_phase = ckpt.get('phase', '?')
    saved_iter  = ckpt.get('iteration', '?')
    score_str   = f'{saved_score:.4f}' if saved_score is not None else 'n/a'
    print(f'\nbest checkpoint: val_score={score_str}  phase={saved_phase}  iter={saved_iter}')
    del ckpt
    print('Checkpoint readable.')

# Fold completion status
print('\nAll-fold completion status:')
all_done = True
for f in range(5):
    done = (CHECKPOINT_DIR / f'fold{f}_best.pt').exists()
    mark = '✓' if done else '○'
    print(f'  [{mark}] Fold {f}')
    if not done:
        all_done = False

if all_done:
    print('\nAll 5 folds complete — run evaluation_complete_v3.ipynb for ensemble metrics')
else:
    next_fold = next(f for f in range(5) if not (CHECKPOINT_DIR / f'fold{f}_best.pt').exists())
    print(f'\nNext: set FOLD={next_fold} in Cell 2 and re-run')
